In [ ]:
# ============================================================
#  SCVELO BENCHMARK TEMPLATE — FIXED EMBEDDING + EPSILON JITTER
# ============================================================

import numpy as np
import anndata
import scvelo as scv
import matplotlib.pyplot as plt

# -----------------------------
# GLOBAL STORAGE DICTIONARY
# -----------------------------
try:
    scvelo_results
except NameError:
    scvelo_results = {}

# -----------------------------
# DATASET REGISTRY
# -----------------------------
DATASETS = {
    "cell_cycle": {
        "X": "./data/real_data_benchmark/cell_cycle/X_cc.npy",
        "V": "./data/real_data_benchmark/cell_cycle/V_cc.npy",
        "color": "./data/real_data_benchmark/cell_cycle/color_cell_cycle_relativePos.npy",
        "embedding": "./data/real_data_benchmark/cell_cycle/flowmap_embedding.npy",
    },
    "pancreas": {
        "X": "./data/real_data_benchmark/pancreas/X_pca.npy",
        "V": "./data/real_data_benchmark/pancreas/V_pca_stochastic.npy",
        "color": "./data/real_data_benchmark/pancreas/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/pancreas/scvelo_umap_embedding.npy",
    },
    "dentate_gyrus": {
        "X": "./data/real_data_benchmark/dentate_gyrus/X_pca.npy",
        "V": "./data/real_data_benchmark/dentate_gyrus/V_pca_dynamical.npy",
        "color": "./data/real_data_benchmark/dentate_gyrus/pseudotime.npy",
        "embedding": "./data/real_data_benchmark/dentate_gyrus/scvelo_umap_embedding.npy",
    },
    "larry": {
        "X": "./data/real_data_benchmark/larry/X_raw.npy",
        "V": "./data/real_data_benchmark/larry/V_raw.npy",
        "color": "./data/real_data_benchmark/larry/distance_pseudotime.npy",
        "embedding": "./data/real_data_benchmark/larry/flowmap_embedding.npy",
    },
}

# -----------------------------
# SELECT DATASET
# -----------------------------
dataset_name = "cell_cycle"   # <<< CHANGE ONLY THIS
cfg = DATASETS[dataset_name]

# -----------------------------
# LOAD MATRICES
# -----------------------------
X = np.load(cfg["X"])
V = np.load(cfg["V"])
color = np.load(cfg["color"])
embedding = np.load(cfg["embedding"])

# -----------------------------
# BUILD ANNADATA
# -----------------------------
adata = anndata.AnnData(X)
adata.layers["position"] = X
adata.layers["velocity"] = V
adata.obs["color"] = np.asarray(color, dtype=float)
adata.obsm["X_umap"] = embedding.copy()   # fixed embedding

# -----------------------------
# SCVELO VECTOR FIELD
# -----------------------------
scv.pp.neighbors(adata, n_neighbors=30, use_rep="X")
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")
scv.tl.velocity_embedding(adata, basis="umap")

# -----------------------------
# 🔧 EPSILON JITTER (DETERMINISTIC)
# -----------------------------
np.random.seed(0)
eps = 1e-6

adata.obsm["X_umap"] = adata.obsm["X_umap"] + eps * np.random.randn(*adata.obsm["X_umap"].shape)
adata.obsm["velocity_umap"] = adata.obsm["velocity_umap"] + eps * np.random.randn(
    *adata.obsm["velocity_umap"].shape
)

# -----------------------------
# PLOT (FINAL, PAPER-LOCKED)
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 5))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",
    color_map="viridis",   # <-- NOT reversed
    arrow_size=1.5,
    linewidth=2.5,
    density=0.5,           # <-- sparser streamlines
    size=660,              # <-- larger dots
    alpha=0.15,            # <-- lighter points
    legend_loc=None,
    colorbar=False,
    show=False,
    ax=ax,
)

ax.set_title(f"")
ax.set_axis_off()
plt.tight_layout()

plt.show()

In [ ]:
dataset_name = "pancreas"   # change this ONLY

cfg = DATASETS[dataset_name]

# -----------------------------
# LOAD PREPARED MATRICES
# -----------------------------
X = np.load(cfg["X"])
V = np.load(cfg["V"])
color = np.load(cfg["color"])
embedding = np.load(cfg["embedding"])

adata = anndata.AnnData(X)
adata.layers["position"] = X
adata.layers["velocity"] = V
adata.obs["color"] = color
adata.obsm["X_umap"] = embedding   # <-- FIXED 2D embedding

# -----------------------------
# SCVELO VECTOR FIELD
# -----------------------------
scv.pp.neighbors(adata, n_neighbors=30, use_rep="X")
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")
scv.tl.velocity_embedding(adata, basis="umap")

# -----------------------------
# SAVE RESULT INTO DICTIONARY
# -----------------------------
scvelo_results[dataset_name] = adata
print(f"\nStored scVelo AnnData for: {dataset_name}")

# -----------------------------
# PLOT (quick check)
# -----------------------------
fig, ax = plt.subplots(figsize=(7, 4))
# scv.pl.velocity_embedding_stream(
#     adata,
#     basis="umap",
#     color="color",
#     cmap="viridis",
#     ax=ax,
#     show=False,
#     density=0.6,
#     arrow_size=2.5,
#     linewidth=2.0,
# )

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",      # dummy continuous
    cmap="viridis",
    colorbar=False,
    legend_loc=None,
    ax=ax,
    show=False,
    size=200,
    alpha=0.15,
    density=1.2,
    linewidth=1.5,
    arrowsize=2.0
)

ax.set_title(f"")
plt.show()

In [ ]:
dataset_name = "dentate_gyrus"   # change this ONLY

cfg = DATASETS[dataset_name]

# -----------------------------
# LOAD PREPARED MATRICES
# -----------------------------
X = np.load(cfg["X"])
V = np.load(cfg["V"])
color = np.load(cfg["color"])
embedding = np.load(cfg["embedding"])

# ---- FIX: set NaN color to 0 ----
color = np.asarray(color, dtype=float).ravel()
color = np.nan_to_num(color, nan=0.0)

adata = anndata.AnnData(X)
adata.layers["position"] = X
adata.layers["velocity"] = V
adata.obs["color"] = color
adata.obsm["X_umap"] = embedding   # <-- FIXED 2D embedding

# -----------------------------
# SCVELO VECTOR FIELD
# -----------------------------
scv.pp.neighbors(adata, n_neighbors=30, use_rep="X")
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")
scv.tl.velocity_embedding(adata, basis="umap")

# -----------------------------
# SAVE RESULT INTO DICTIONARY
# -----------------------------
scvelo_results[dataset_name] = adata
print(f"\nStored scVelo AnnData for: {dataset_name}")

# -----------------------------
# PLOT (quick check)
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 6))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",
    cmap="viridis",
    colorbar=False,
    legend_loc=None,
    ax=ax,
    show=False,
    size=200,
    alpha=0.1,
    density=1.2,
    linewidth=2.0,
    arrowsize=1.5,
)

ax.set_title("")
plt.show()

In [ ]:
# ============================================================
#  LARRY DATA — FIXED FLOWMAP EMBEDDING + PCA FOR VECTOR FIELD
#  Saves final AnnData into scvelo_results["larry"]
# ============================================================

from sklearn.decomposition import PCA

# -----------------------------
# GLOBAL STORAGE DICT
# -----------------------------
try:
    scvelo_results
except NameError:
    scvelo_results = {}

# -----------------------------
# SELECT DATASET
# -----------------------------
dataset_name = "larry"

cfg = DATASETS[dataset_name]

# -----------------------------
# LOAD RAW FLOWMAP-PREPARED DATA
# -----------------------------
X = np.load(cfg["X"])             # raw gene expression
V = np.load(cfg["V"])             # raw velocity
color = np.load(cfg["color"])
embedding = np.load(cfg["embedding"])  # FlowMap embedding

print(f"Loaded shapes: X={X.shape}, V={V.shape}, emb={embedding.shape}")

adata = anndata.AnnData(X)
adata.layers["position"] = X
adata.layers["velocity"] = V
adata.obsm["X_umap"] = embedding    # ← FIXED 2D embedding

# -----------------------------
# COLOR: HARD NORMALIZE TO [0, 1]
# -----------------------------
color = np.asarray(color, dtype=float)
color = np.nan_to_num(color, nan=0.0)

cmin, cmax = color.min(), color.max()
if cmax > cmin:
    color = (color - cmin) / (cmax - cmin)
else:
    color = np.zeros_like(color)

# SAFETY CHECK
assert np.isfinite(color).all()
assert color.min() >= 0.0 and color.max() <= 1.0

adata.obs["color"] = color   # <-- DO NOT CAST TO STR


# -----------------------------
# PCA (same dimensionality as others)
# -----------------------------
n_pcs = 30
print(f"Running PCA ({n_pcs} PCs)...")

pca = PCA(n_components=n_pcs, random_state=0)
X_pca = pca.fit_transform(X)
adata.obsm["X_pca"] = X_pca

# Project velocity: V_pca = V @ PCs
components = pca.components_.T    # (genes × PCs)
V_pca = V @ components
adata.obsm["velocity_pca"] = V_pca

print("X_pca:", X_pca.shape)
print("V_pca:", V_pca.shape)

# -----------------------------
# SCVELO VECTOR FIELD
# -----------------------------
scv.pp.neighbors(adata, n_neighbors=30, use_rep="X_pca")

# Build graph in high-dim velocity space
scv.tl.velocity_graph(adata, xkey="position", vkey="velocity")

# Project velocity field INTO the fixed FlowMap embedding
scv.tl.velocity_embedding(adata, basis="umap", vkey="velocity")

print("Velocity embedding fields added:")
print([k for k in adata.obsm.keys() if "velocity" in k])

# -----------------------------
# SAVE INTO GLOBAL DICT
# -----------------------------
scvelo_results[dataset_name] = adata
print(f"\nSaved AnnData object to scvelo_results['{dataset_name}']")

# -----------------------------
# OPTIONAL PLOT
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 5))
scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",
    cmap="viridis",
    ax=ax,
    colorbar=False,
    show=False,
    density=1.2,
    arrow_size=1.0,
    linewidth=0.6,
    alpha=0.7,
)
ax.set_title(f"")
plt.show()


In [ ]:
# -----------------------------
# LOAD RAW FLOWMAP-PREPARED DATA
# -----------------------------
X = np.load(cfg["X"])             # raw gene expression
V = np.load(cfg["V"])             # raw velocity
color = np.load(cfg["color"])     # <-- use AS IS
embedding = np.load(cfg["embedding"])  # FlowMap embedding

print(f"Loaded shapes: X={X.shape}, V={V.shape}, emb={embedding.shape}")

# adata = anndata.AnnData(X)
# adata.layers["position"] = X
# adata.layers["velocity"] = V
# adata.obsm["X_umap"] = embedding    # ← FIXED 2D embedding

# -----------------------------
# COLOR: CLIP UPPER 95% QUANTILE
# -----------------------------
color = np.asarray(color, dtype=float)
color = np.nan_to_num(color, nan=0.0, posinf=0.0, neginf=0.0)

# clip only the top tail
q95 = np.quantile(color, 0.95)
color = np.clip(color, None, q95)

adata.obs["color"] = color


# sanity (no normalization!)
assert np.isfinite(color).all()
assert color.shape[0] == adata.n_obs

adata.obs["color"] = color   # <-- continuous, unscaled



fig, ax = plt.subplots(figsize=(6, 5))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",
    color_map="viridis",  # <-- red-shifted
    colorbar=False,
    legend_loc=None,
    ax=ax,
    show=False,
    size=100,
    alpha=0.05,
    density=1.2,
    linewidth=2.0,
    arrowsize=1.5,
)

ax.set_title("")
plt.show()